In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. LOAD & CLEAN DATA
df = pd.read_csv('/content/drive/MyDrive/Dav Datasets/Diseertaion dataset/Brent Oil Futures Historical Data (1).csv')

df = df[['Date', 'Price']]
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Ensure numeric
df['Price'] = df['Price'].replace(',', '', regex=True).astype(float)

# 2. FEATURE ENGINEERING 

# Lag features (past values)
df['lag1'] = df['Price'].shift(1)
df['lag7'] = df['Price'].shift(7)
df['lag30'] = df['Price'].shift(30)

# Rolling statistics
df['rolling_mean_7'] = df['Price'].rolling(window=7).mean()
df['rolling_std_7'] = df['Price'].rolling(window=7).std()

# Time-based features
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month
df['year'] = df.index.year

# Target = future price (next day)
df['target'] = df['Price'].shift(-1)

# Drop missing values from shifting
df.dropna(inplace=True)

# 3. TRAIN / TEST SPLIT
FEATURES = ['lag1', 'lag7', 'lag30',
            'rolling_mean_7', 'rolling_std_7',
            'dayofweek', 'month', 'year']

TARGET = 'target'

tss = TimeSeriesSplit(n_splits=5, test_size=365)

scores = []
all_preds = []
all_actuals = []

# 4. MODEL TRAINING (XGBOOST)
for train_idx, val_idx in tss.split(df):
    train = df.iloc[train_idx]
    test = df.iloc[val_idx]

    X_train = train[FEATURES]
    y_train = train[TARGET]

    X_test = test[FEATURES]
    y_test = test[TARGET]

    model = xgb.XGBRegressor(
        objective='reg:squarederror',   # fixed (was outdated)
        n_estimators=500,
        learning_rate=0.01,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    preds = model.predict(X_test)

    # Store results
    all_preds.extend(preds)
    all_actuals.extend(y_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    scores.append(rmse)

# 5. EVALUATION
mae = mean_absolute_error(all_actuals, all_preds)
mse = mean_squared_error(all_actuals, all_preds)
rmse = np.sqrt(mse)
r2 = r2_score(all_actuals, all_preds)

print("Cross-validation RMSE:", np.mean(scores))
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# 6. VISUALIZATION
results_df = pd.DataFrame({
    'Actual': all_actuals,
    'Predicted': all_preds
})

results_df.plot(figsize=(12,6), title="Actual vs Predicted Prices")
plt.show()